# Diurnal Hourly Profiles
## Is the school-term–vacation difference concentrated at arrival or present throughout the day?

This diagnostic shows the complete diurnal cycle. An arrival-related association should be especially apparent around 07:00–09:00. A similar separation at all hours, including before dawn, suggests a broader background difference, such as seasonality or general activity. Hours without school activity provide reference windows; they do not automatically establish a valid counterfactual.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Diagnostic: hourly profiles, the diurnal curves of Fig. 1 of the article and the hour-by-hour simple difference used to examine the control window. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']
colores_contaminantes = {'CO': '#3498db', 'NO2': '#e74c3c', 'O3': '#2ecc71',
                         'PM10': '#f39c12', 'PM2.5': '#9b59b6'}
pisos_fisicos = {'CO': 0.05, 'NO2': 0.05, 'O3': 0.5, 'PM10': 1.0, 'PM2.5': 0.5}

output_dir = Path('../data_processed')
perfil_dir = output_dir / 'perfil_horario'
perfil_dir.mkdir(exist_ok=True)

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'datos_horarios_imputados_hibrido.csv.gz',
                       output_dir / 'datos_horarios_imputados_MIMA.csv.gz',
                       output_dir / 'gaps_franjas_hibrido.csv',
                       output_dir / 'calendario_oficial_sep.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 4 files


## 2. Loading hourly data and building the hour-specific panel

In [2]:
def cargar_panel_horario(ruta, nombre_dataset):
    '\n    Load the imputed hourly series, pivot pollutants to columns,\n    and merge the official calendar and calendar features.\n    '
    horario = pd.read_csv(ruta, parse_dates=['date'])
    horario['fecha'] = horario['date'].dt.normalize()
    horario['hora'] = horario['date'].dt.hour
    ancho = horario.pivot_table(index=['fecha', 'hora', 'estacion'],
                                columns='parametro', values='valor').reset_index()

    calendario = pd.read_csv('../data_processed/calendario_oficial_sep.csv', parse_dates=['date'])
    mapa_clase = calendario.set_index('date')['lectivo']
    ancho['clase'] = ancho['fecha'].map(mapa_clase)
    ancho = ancho.dropna(subset=['clase'] + contaminantes)
    ancho['clase'] = ancho['clase'].astype(int)
    ancho['dow'] = ancho['fecha'].dt.dayofweek
    ancho['mes'] = ancho['fecha'].dt.month
    iso = ancho['fecha'].dt.isocalendar()
    ancho['wk'] = iso.year.astype(str) + '-' + iso.week.astype(str).str.zfill(2)
    for c in contaminantes:
        ancho[c] = ancho[c].clip(lower=pisos_fisicos[c])
        ancho[f'l_{c}'] = np.log(ancho[c])
    print(f"[{nombre_dataset}] hourly panel: {len(ancho):,} hour-day-unit rows")
    return ancho

panel_horario = cargar_panel_horario('../data_processed/datos_horarios_imputados_hibrido.csv.gz', 'hibrido')

# Weekdays only: compare school-term weekdays with vacation weekdays
entre_semana = panel_horario[panel_horario['dow'] < 5].copy()
print(f"Weekdays: {entre_semana['fecha'].nunique()} fechas "
      f"({int((entre_semana.groupby('fecha')['clase'].first() == 1).sum())} during school term, "
      f"{int((entre_semana.groupby('fecha')['clase'].first() == 0).sum())} during vacation)")

[hibrido] hourly panel: 157,824 hour-day-unit rows
Weekdays: 783 fechas (628 during school term, 155 during vacation)


## 3. Weekday diurnal curves: school term versus vacation

In [3]:
curvas = entre_semana.groupby(['hora', 'clase'])[contaminantes].mean().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.flatten()

for i, c in enumerate(contaminantes):
    ax = axes[i]
    ax.axvspan(7, 9, color='#f1c40f', alpha=0.25)
    ax.axvspan(10, 12, color='#95a5a6', alpha=0.18)
    for cl, color, etiqueta in [(0, '#3498db', 'Vacation (weekdays)'), (1, '#e74c3c', 'School term (weekdays)')]:
        sub = curvas[curvas['clase'] == cl]
        ax.plot(sub['hora'], sub[c], marker='o', markersize=3.5, color=color,
                label=etiqueta, linewidth=2)
    ax.set_title(c, fontsize=12, fontweight='bold')
    ax.set_xticks(range(0, 24, 3))
    if i >= 3:
        ax.set_xlabel('Hour of day', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Mean level', fontsize=11, fontweight='bold')
axes[3].set_ylabel('Mean level', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[5].axis('off')
axes[5].text(0.03, 0.8, 'Yellow: arrival window, 7-9.\nGray: control window, 10-12.\n\nDifferences at all hours, including\nnighttime, can reflect seasonal\nbackground and other activity.\nThe question is whether the gap\nwidens further during 7-9.\nSection 8 summarizes the raw\nhourly and window contrasts.',
             fontsize=10.5, va='top')
fig.suptitle('Weekday diurnal cycle: school term vs vacation (hybrid imputation)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(perfil_dir / '01_curvas_diurnas.png', dpi=150, bbox_inches='tight')
plt.show()

curvas.to_csv(perfil_dir / 'curvas_diurnas_por_regimen.csv', index=False)
print(f"Saved to: {perfil_dir / 'curvas_diurnas_por_regimen.csv'}")
print(f"Plot saved to: {perfil_dir / '01_curvas_diurnas.png'}")

Saved to: ../data_processed/perfil_horario/curvas_diurnas_por_regimen.csv
Plot saved to: ../data_processed/perfil_horario/01_curvas_diurnas.png


## 4. Adjusted school-term association HOUR BY HOUR

For each hour, fit the same level regression used elsewhere: log concentration on school-term status, monitoring-unit, month and day-of-week effects, with weekly-clustered errors and weekdays only. Pre-dawn and night hours provide reference windows where an arrival-specific association would be expected to be small.

### Hour-specific equation

Estimate separately for each $h\in\{0,\ldots,23\}$:

$$ \log y^{(h)}_{st}=\alpha_h+\beta_h\text{School}_t+\eta_s+\gamma_{m(t)}+\delta_{d(t)}+\varepsilon^{(h)}_{st}. $$

Plot $(e^{\beta_h}-1)\times100$ and its confidence interval. The profile examines consistency across 24 windows: an arrival-related pattern would concentrate around labels 7 and 8, with coefficients closer to zero before dawn. These are separate level regressions, not hourly double-difference estimates.

In [4]:
def contraste_en_hora(df_hora, columna_y):
    '\n    OLS log(y) ~ clase + zone + month + weekday, with weekly-clustered errors,\n    using observations from ONE hour of the day.\n    '
    df = df_hora.dropna(subset=[columna_y]).copy()
    X = pd.concat([pd.Series(1.0, index=df.index, name='const'), df['clase'].astype(float),
                   pd.get_dummies(df['estacion'], prefix='est', drop_first=True).astype(float),
                   pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float),
                   pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float)], axis=1)
    Xv = X.values
    n, k = Xv.shape
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    y = df[columna_y].values
    beta_todos = XtX_inv @ Xv.T @ y
    residuos = y - Xv @ beta_todos
    grupos = pd.factorize(df['wk'])[0]
    orden = np.argsort(grupos, kind='stable')
    Xo, ro, go = Xv[orden], residuos[orden], grupos[orden]
    inicios = np.flatnonzero(np.r_[True, go[1:] != go[:-1]])
    G = len(inicios)
    fines = np.r_[inicios[1:], len(go)]
    S = np.zeros((k, k))
    for s, e in zip(inicios, fines):
        zg = Xo[s:e].T @ ro[s:e]
        S += np.outer(zg, zg)
    V = XtX_inv @ S @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    idx = X.columns.get_loc('clase')
    beta, se = beta_todos[idx], np.sqrt(V[idx, idx])
    return beta, se

filas_horas = []
for hora in range(24):
    df_hora = entre_semana[entre_semana['hora'] == hora]
    for c in contaminantes:
        beta, se = contraste_en_hora(df_hora, f'l_{c}')
        filas_horas.append({'hora': hora, 'contaminante': c,
                            'pct': (np.exp(beta) - 1) * 100,
                            'ci_lo_pct': (np.exp(beta - 1.96 * se) - 1) * 100,
                            'ci_hi_pct': (np.exp(beta + 1.96 * se) - 1) * 100,
                            'p': 2 * (1 - stats.norm.cdf(abs(beta / se)))})
tabla_horas = pd.DataFrame(filas_horas)
tabla_horas[['pct', 'ci_lo_pct', 'ci_hi_pct']] = tabla_horas[['pct', 'ci_lo_pct', 'ci_hi_pct']].round(2)
tabla_horas['p'] = tabla_horas['p'].round(4)
tabla_horas.to_csv(perfil_dir / 'efecto_clase_por_hora.csv', index=False)
print(f"Saved to: {perfil_dir / 'efecto_clase_por_hora.csv'}")

print('\nADJUSTED SCHOOL-TERM EFFECT BY HOUR (CO):')
print(tabla_horas.query("contaminante == 'CO'")[['hora', 'pct', 'ci_lo_pct', 'ci_hi_pct', 'p']].to_string(index=False))

Saved to: ../data_processed/perfil_horario/efecto_clase_por_hora.csv

ADJUSTED SCHOOL-TERM EFFECT BY HOUR (CO):
 hora   pct  ci_lo_pct  ci_hi_pct      p
    0 -0.80     -13.37      13.58 0.9071
    1 -0.70     -13.16      13.54 0.9177
    2 -0.38     -12.94      13.99 0.9562
    3  0.21     -12.50      14.77 0.9755
    4  0.38     -12.35      14.97 0.9559
    5  1.21     -11.54      15.79 0.8614
    6  3.67      -8.77      17.81 0.5805
    7  6.76      -5.53      20.64 0.2946
    8 10.52      -1.57      24.11 0.0907
    9  7.41      -4.14      20.36 0.2180
   10  4.02      -7.29      16.71 0.5021
   11  2.16      -8.94      14.60 0.7161
   12  2.02      -9.35      14.81 0.7404
   13  2.54      -9.25      15.86 0.6872
   14  2.41      -9.62      16.03 0.7092
   15  2.51      -9.89      16.62 0.7060
   16  2.40     -10.24      16.81 0.7241
   17  2.73     -10.13      17.43 0.6927
   18  2.67     -10.22      17.42 0.7001
   19  2.89      -9.70      17.24 0.6688
   20  2.33      -9.67     

In [5]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.flatten()

for i, c in enumerate(contaminantes):
    ax = axes[i]
    sub = tabla_horas[tabla_horas['contaminante'] == c]
    ax.axvspan(7, 9, color='#f1c40f', alpha=0.25)
    ax.fill_between(sub['hora'], sub['ci_lo_pct'], sub['ci_hi_pct'],
                    color=colores_contaminantes[c], alpha=0.18)
    ax.plot(sub['hora'], sub['pct'], marker='o', markersize=3.5,
            color=colores_contaminantes[c], linewidth=2)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(c, fontsize=12, fontweight='bold')
    ax.set_xticks(range(0, 24, 3))
    if i >= 3:
        ax.set_xlabel('Hour of day', fontsize=11, fontweight='bold')
axes[0].set_ylabel('School-term effect (%)', fontsize=11, fontweight='bold')
axes[3].set_ylabel('School-term effect (%)', fontsize=11, fontweight='bold')
axes[5].axis('off')
hora_max_co = int(tabla_horas[tabla_horas['contaminante'] == 'CO'].set_index('hora')['pct'].idxmax())
axes[5].text(0.03, 0.78, f'The adjusted CO point-estimate\nprofile reaches its maximum at {hora_max_co:02d}:00\n(arrival window: 07-09).\n\nHourly confidence intervals are\nwide. Formal inference uses the\nwithin-day window design; this\nplot describes the profile and\nthe location of its peak.',
             fontsize=10.5, va='top')
fig.suptitle('Hourly school-term association (percent effect with transformed 95% CI; weekdays; weekly-clustered errors)\nHours outside school arrival serve as reference windows',
             fontsize=13.5, fontweight='bold')
plt.tight_layout()
plt.savefig(perfil_dir / '02_efecto_por_hora.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {perfil_dir / '02_efecto_por_hora.png'}")

Plot saved to: ../data_processed/perfil_horario/02_efecto_por_hora.png


## 5. Sensitivity to MIMA imputation: hourly profile

In [6]:
panel_horario_mima = cargar_panel_horario('../data_processed/datos_horarios_imputados_MIMA.csv.gz', 'MIMA')
entre_semana_mima = panel_horario_mima[panel_horario_mima['dow'] < 5]

trayectoria_mima = []
for hora in range(24):
    beta, se = contraste_en_hora(entre_semana_mima[entre_semana_mima['hora'] == hora], 'l_CO')
    trayectoria_mima.append((np.exp(beta) - 1) * 100)
trayectoria_hibrido = tabla_horas.query("contaminante == 'CO'")['pct'].values
correlacion_trayectorias = np.corrcoef(trayectoria_hibrido, trayectoria_mima)[0, 1]

print("=" * 70)
print('MIMA CHECK: HOURLY CO EFFECT PROFILE')
print("=" * 70)
print(f"Correlation between hybrid and MIMA profiles (24 hours): {correlacion_trayectorias:.3f}")
print(f"Hour 7 — hybrid: {trayectoria_hibrido[7]:+.1f}% | MIMA: {trayectoria_mima[7]:+.1f}%")
print(f"Hour 8 — hybrid: {trayectoria_hibrido[8]:+.1f}% | MIMA: {trayectoria_mima[8]:+.1f}%")

[MIMA] hourly panel: 157,824 hour-day-unit rows


MIMA CHECK: HOURLY CO EFFECT PROFILE
Correlation between hybrid and MIMA profiles (24 hours): 0.997
Hour 7 — hybrid: +6.8% | MIMA: +6.1%
Hour 8 — hybrid: +10.5% | MIMA: +10.0%


## 6. Heterogeneity across monitoring zones

If the CO association reflects arrival-related traffic, it might be more apparent at traffic-oriented or central locations than peripheral ones. We estimate the within-day gap model separately by monitoring zone for CO and NO2. Without direct traffic and school-location data, this remains a diagnostic rather than source attribution.

In [7]:
gaps = pd.read_csv('../data_processed/gaps_franjas_hibrido.csv', parse_dates=['date'])

def contraste_gap_estacion(df_estacion, columna_y):
    '\n    gap ~ clase + month + weekday, with weekly-clustered errors, within ONE unit.\n    '
    df = df_estacion.dropna(subset=[columna_y]).copy()
    X = pd.concat([pd.Series(1.0, index=df.index, name='const'), df['clase'].astype(float),
                   pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float),
                   pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float)], axis=1)
    Xv = X.values
    n, k = Xv.shape
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    y = df[columna_y].values
    beta_todos = XtX_inv @ Xv.T @ y
    residuos = y - Xv @ beta_todos
    grupos = pd.factorize(df['wk'])[0]
    orden = np.argsort(grupos, kind='stable')
    Xo, ro, go = Xv[orden], residuos[orden], grupos[orden]
    inicios = np.flatnonzero(np.r_[True, go[1:] != go[:-1]])
    G = len(inicios)
    fines = np.r_[inicios[1:], len(go)]
    S = np.zeros((k, k))
    for s, e in zip(inicios, fines):
        zg = Xo[s:e].T @ ro[s:e]
        S += np.outer(zg, zg)
    V = XtX_inv @ S @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    idx = X.columns.get_loc('clase')
    beta, se = beta_todos[idx], np.sqrt(V[idx, idx])
    return beta, se, n

filas_estaciones = []
for estacion_m in sorted(gaps['estacion'].unique()):
    sub = gaps[gaps['estacion'] == estacion_m]
    for c in ['CO', 'NO2']:
        beta, se, n = contraste_gap_estacion(sub, f'gap_{c}')
        filas_estaciones.append({'estacion': estacion_m, 'contaminante': c,
                                 'pct': round((np.exp(beta) - 1) * 100, 2),
                                 'ci_lo_pct': round((np.exp(beta - 1.96 * se) - 1) * 100, 2),
                                 'ci_hi_pct': round((np.exp(beta + 1.96 * se) - 1) * 100, 2),
                                 'p': round(2 * (1 - stats.norm.cdf(abs(beta / se))), 4), 'n': n})
tabla_estaciones = pd.DataFrame(filas_estaciones)

print("=" * 70)
print('GAP DiD BY MONITORING UNIT (CO AND NO2)')
print("=" * 70)
print(tabla_estaciones.to_string(index=False))
tabla_estaciones.to_csv(perfil_dir / 'gap_did_por_estacion.csv', index=False)
print(f"\nSaved to: {perfil_dir / 'gap_did_por_estacion.csv'}")

GAP DiD BY MONITORING UNIT (CO AND NO2)
 estacion contaminante   pct  ci_lo_pct  ci_hi_pct      p    n
   CENTRO           CO  1.87      -0.87       4.69 0.1836 1096
   CENTRO          NO2  6.60      -1.87      15.80 0.1305 1096
 NORESTE3           CO  4.61       1.38       7.95 0.0049 1096
 NORESTE3          NO2 -1.35      -9.13       7.10 0.7463 1096
NOROESTE3           CO  3.38      -1.39       8.38 0.1680 1096
NOROESTE3          NO2 -1.70     -10.24       7.64 0.7109 1096
   NORTE2           CO  6.67       1.59      12.01 0.0095 1096
   NORTE2          NO2 -1.68      -9.62       6.95 0.6932 1096
      SUR           CO  2.91      -0.26       6.18 0.0727 1096
      SUR          NO2  4.69      -3.53      13.60 0.2719 1096
SUROESTE2           CO  8.03       2.25      14.13 0.0059 1096
SUROESTE2          NO2 -0.10      -8.63       9.22 0.9820 1096

Saved to: ../data_processed/perfil_horario/gap_did_por_estacion.csv


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=False)

for ax, c in zip(axes, ['CO', 'NO2']):
    sub = tabla_estaciones[tabla_estaciones['contaminante'] == c].reset_index(drop=True)
    y_pos = np.arange(len(sub))
    ax.errorbar(sub['pct'], y_pos,
                xerr=[sub['pct'] - sub['ci_lo_pct'], sub['ci_hi_pct'] - sub['pct']],
                fmt='o', color=colores_contaminantes[c], markersize=7,
                capsize=4, linewidth=2)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['estacion'])
    ax.set_title(f'Gap DiD by monitoring unit: {c}', fontsize=12, fontweight='bold')
    ax.set_xlabel('School-term change in the morning/control ratio (%)', fontsize=10.5, fontweight='bold')

fig.suptitle('Spatial heterogeneity (transformed 95% CI; month and weekday controls; weekly-clustered errors)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(perfil_dir / '03_gap_por_estacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {perfil_dir / '03_gap_por_estacion.png'}")

Plot saved to: ../data_processed/perfil_horario/03_gap_por_estacion.png


## 7. Particulate levels relative to the WHO 2021 guidelines

For PM10 and PM2.5, monitored in µg/m³, compare all-hour study-period averages by monitoring unit with the WHO annual reference values: 5 µg/m³ for PM2.5 and 15 µg/m³ for PM10. This contextual comparison does not estimate individual health effects.

In [9]:
guias_oms = {'PM10': 15.0, 'PM2.5': 5.0}
promedios_pm = (panel_horario.groupby('estacion')[['PM10', 'PM2.5']].mean().round(1))
tabla_oms = promedios_pm.copy()
for c in ['PM10', 'PM2.5']:
    tabla_oms[f'{c}_veces_guia'] = (promedios_pm[c] / guias_oms[c]).round(1)

print("=" * 70)
print('FULL-PERIOD MEANS VS WHO ANNUAL GUIDELINES (2021)')
print("=" * 70)
print(tabla_oms)
print(f"\nMetropolitan mean: PM2.5 = {panel_horario['PM2.5'].mean():.1f} µg/m³ "
      f"({panel_horario['PM2.5'].mean()/guias_oms['PM2.5']:.1f}x the WHO annual guideline of 5 µg/m³)")
print(f"                        PM10  = {panel_horario['PM10'].mean():.1f} µg/m³ "
      f"({panel_horario['PM10'].mean()/guias_oms['PM10']:.1f}x the WHO annual guideline of 15 µg/m³)")
print('\nNote: pooled 2023-2025 means from imputed data; descriptive health context,')
print("not annual compliance estimates or part of the paper's identification strategy.")

tabla_oms.to_csv(perfil_dir / 'niveles_pm_vs_guia_oms.csv')
print(f"\nSaved to: {perfil_dir / 'niveles_pm_vs_guia_oms.csv'}")

FULL-PERIOD MEANS VS WHO ANNUAL GUIDELINES (2021)
parametro  PM10  PM2.5  PM10_veces_guia  PM2.5_veces_guia
estacion                                                 
CENTRO     56.3   22.0              3.8               4.4
NORESTE3   55.8   20.9              3.7               4.2
NOROESTE3  56.1   16.8              3.7               3.4
NORTE2     61.0   18.0              4.1               3.6
SUR        56.4   19.9              3.8               4.0
SUROESTE2  59.5   21.6              4.0               4.3

Metropolitan mean: PM2.5 = 19.9 µg/m³ (4.0x the WHO annual guideline of 5 µg/m³)
                        PM10  = 57.5 µg/m³ (3.8x the WHO annual guideline of 15 µg/m³)

Note: pooled 2023-2025 means from imputed data; descriptive health context,
not annual compliance estimates or part of the paper's identification strategy.

Saved to: ../data_processed/perfil_horario/niveles_pm_vs_guia_oms.csv


## 8. Area between the diurnal curves

Curve separation is not unique to CO; other pollutants can also differ by regime. We examine whether separation widens near arrival, quantifying hourly differences and window-specific areas in concentration units multiplied by hours and in percentages. Comparing arrival, mid-morning and pre-dawn windows is a descriptive complement to the adjusted within-day analysis, not an identical estimator or a direct personal-exposure measurement.

### Window areas and within-day comparisons

The unadjusted relative difference at hour $h$ is $e_h=\bar y_h^{\text{school term}}/\bar y_h^{\text{vacation}}-1$, multiplied by 100 for percentages. For one-hour records, the area within a window is

$$ A_{[h_1,h_2]}=\sum_{h=h_1}^{h_2}\left(\bar y_h^{\text{school term}}-\bar y_h^{\text{vacation}}\right), $$

in concentration units multiplied by hours. Differences between windows describe additional morning separation relative to the background. They are unadjusted and are not algebraically identical to the log-gap coefficient from notebook 05, which adjusts for calendar composition.

In [10]:
ventanas = {'madrugada_0_5': range(0, 6), 'pico_7_9': [7, 8], 'control_10_12': [10, 11],
            'tarde_13_18': range(13, 19), 'noche_19_23': range(19, 24)}

espacio_por_hora = {}
area_filas = []
for c in contaminantes:
    ancho_c = curvas.pivot(index='hora', columns='clase', values=c)
    espacio_pct = (ancho_c[1] / ancho_c[0] - 1) * 100
    espacio_abs = ancho_c[1] - ancho_c[0]
    espacio_por_hora[c] = espacio_pct
    for nombre_v, horas_v in ventanas.items():
        horas_v = list(horas_v)
        area_filas.append({'contaminante': c, 'ventana': nombre_v,
                           'espacio_promedio_pct': round(espacio_pct.loc[horas_v].mean(), 1),
                           'area_unidades_h': round(espacio_abs.loc[horas_v].sum(), 2),
                           'horas': len(horas_v)})

tabla_espacio_hora = pd.DataFrame(espacio_por_hora).round(1)
tabla_espacio_hora.to_csv(perfil_dir / 'espacio_crudo_por_hora.csv')
tabla_areas = pd.DataFrame(area_filas)
tabla_areas.to_csv(perfil_dir / 'espacio_por_ventana.csv', index=False)

print("=" * 70)
print('RAW PERCENTAGE SEPARATION BETWEEN CURVES, AVERAGED BY WINDOW')
print("=" * 70)
print(tabla_areas.pivot(index='contaminante', columns='ventana', values='espacio_promedio_pct')
      [['madrugada_0_5', 'pico_7_9', 'control_10_12', 'tarde_13_18', 'noche_19_23']])

print('\nRAW DIFFERENCE BETWEEN WINDOW CONTRASTS (percentage points):')
print(f"{'':>6} {'pico-control':>14} {'pico-madrugada':>16}")
esp = tabla_areas.pivot(index='contaminante', columns='ventana', values='espacio_promedio_pct')
for c in contaminantes:
    d_control = esp.loc[c, 'pico_7_9'] - esp.loc[c, 'control_10_12']
    d_madrugada = esp.loc[c, 'pico_7_9'] - esp.loc[c, 'madrugada_0_5']
    print(f"{c:>6} {d_control:>+13.1f} {d_madrugada:>+15.1f}")
lideres = [c for c in ['CO', 'NO2', 'PM10', 'PM2.5'] if esp.loc[c, 'pico_7_9'] - esp.loc[c, 'control_10_12'] > 0]
print('\n=> Background differences occur at multiple hours, including nighttime.')
print('   This RAW summary measures additional arrival-window separation;')
print(f"   the arrival-minus-control separation is positive for: {', '.join(lideres) if lideres else 'none'}.")
print('   the formal adjusted log-gap estimate is in notebook 05.')
o3_pico = esp.loc['O3', 'pico_7_9']
o3_control = esp.loc['O3', 'control_10_12']
if o3_pico < o3_control:
    print(f"=> Chemistry: the raw O3 contrast in the arrival window is LOWER than in the")
    print(f"   control window ({o3_pico:+.1f}% at 7-9 vs {o3_control:+.1f}% at 10-12); this pattern")
    print('   could be consistent with NO titration, but these contrasts do not identify its cause.')
else:
    print(f"=> Chemistry: the raw O3 contrast in the arrival window ({o3_pico:+.1f}% at 7-9) is not below")
    print(f"   the control-window contrast ({o3_control:+.1f}% at 10-12); no clear titration pattern here.")
print(f"\nSaved to: {perfil_dir / 'espacio_crudo_por_hora.csv'}")
print(f"Saved to: {perfil_dir / 'espacio_por_ventana.csv'}")

RAW PERCENTAGE SEPARATION BETWEEN CURVES, AVERAGED BY WINDOW
ventana       madrugada_0_5  pico_7_9  control_10_12  tarde_13_18  noche_19_23
contaminante                                                                  
CO                      7.6      16.8           10.5         11.2         10.6
NO2                    21.6      19.5           14.2         23.3         22.8
O3                     -1.1      -1.1           -0.9          2.2         -3.0
PM10                    4.0      13.4           13.8         13.2          7.5
PM2.5                   5.6      16.3            8.6         13.7         10.0

RAW DIFFERENCE BETWEEN WINDOW CONTRASTS (percentage points):
         pico-control   pico-madrugada
    CO          +6.3            +9.2
   NO2          +5.3            -2.1
    O3          -0.2            +0.0
  PM10          -0.4            +9.4
 PM2.5          +7.7           +10.7

=> Background differences occur at multiple hours, including nighttime.
   This RAW summary measure

In [11]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.flatten()

for i, c in enumerate(contaminantes):
    ax = axes[i]
    ancho_c = curvas.pivot(index='hora', columns='clase', values=c)
    ax.axvspan(7, 9, color='#f1c40f', alpha=0.25)
    ax.axvspan(10, 12, color='#95a5a6', alpha=0.18)
    ax.plot(ancho_c.index, ancho_c[0], color='#3498db', linewidth=2, label='Vacation (weekdays)')
    ax.plot(ancho_c.index, ancho_c[1], color='#e74c3c', linewidth=2, label='School term (weekdays)')
    ax.fill_between(ancho_c.index, ancho_c[0], ancho_c[1],
                    where=(ancho_c[1] >= ancho_c[0]), color='#e74c3c', alpha=0.22, interpolate=True)
    ax.fill_between(ancho_c.index, ancho_c[0], ancho_c[1],
                    where=(ancho_c[1] < ancho_c[0]), color='#3498db', alpha=0.22, interpolate=True)
    pico = esp.loc[c, 'pico_7_9']
    fondo = esp.loc[c, 'madrugada_0_5']
    ax.set_title(f"{c}  (7-9 contrast: {pico:+.0f}% | 0-5 background: {fondo:+.0f}%)",
                 fontsize=11, fontweight='bold')
    ax.set_xticks(range(0, 24, 3))
    if i >= 3:
        ax.set_xlabel('Hour of day', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Mean level', fontsize=11, fontweight='bold')
axes[3].set_ylabel('Mean level', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[5].axis('off')
axes[5].text(0.03, 0.8, 'Shading shows regime differences\n(red: school term higher;\nblue: school term lower).\n\nPanel titles compare arrival and\nearly-morning raw contrasts.\nTheir difference is descriptive;\nit does not isolate school traffic\nor equal the adjusted log-gap DiD.',
             fontsize=10.5, va='top')
fig.suptitle('Raw diurnal regime differences: background and additional arrival-window separation\n(descriptive curves; formal adjusted inference is reported in notebook 05)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(perfil_dir / '04_espacio_entre_curvas.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {perfil_dir / '04_espacio_entre_curvas.png'}")

print('\nCOMPLETED')

Plot saved to: ../data_processed/perfil_horario/04_espacio_entre_curvas.png

COMPLETED
